# FlashAttention

> 4096 个 Token 的 Attention 分数矩阵是 `4096 × 4096`，约 1678 万个数。标准实现的流程是：算出整块矩阵，写入 HBM，再读回来完成 Softmax 与加权求和——一来一回搬运的数据量，是计算单元真正消费量的许多倍。
>
> FlashAttention 的出发点就在这里：优化的重点不是减少乘法，而是减少 HBM 与片上 SRAM 之间的数据搬运。
> 本附录介绍 FlashAttention 的四组核心内容：
>
> 1. **存储层级**：HBM 容量较大，SRAM 容量较小但访问更快；标准 Attention 的瓶颈在两者之间的数据搬运。
>
> 2. **Online Softmax**：用 running max 与 running sum 分块计算数值稳定的 Softmax，不再依赖完整的一行分数。
>
> 3. **Tiling**：Q、K、V 小块在 SRAM 中完成计算，写回 HBM 前直接累积输出。
>
> 4. **等价与实现**：计算顺序改变但数学目标不变，用简化 PyTorch 实现并验证与标准 Attention 的数值等价。


处理 4096 个 Token 时，会产生一个 `4096×4096` 的注意力矩阵。这个矩阵包含多少分数？4096 乘以 4096 等于 16777216，也就是约 1678 万个分数。

标准实现需要把这块中间结果写入 HBM，随后再读回来完成 Softmax 和与 $V$ 的乘法。换句话说，数据在慢速存储上来回搬运，才能完成一次注意力计算。

那么 FlashAttention 是怎么优化的呢？它不改变 Attention 的数学定义，而是改变计算顺序。具体来说，它把 Q、K、V 分块放入更快的片上 SRAM，并用 Online Softmax 逐块维护最大值、归一化分母和输出累积量。

中间的 $N×N$ 矩阵因此不再完整写回 HBM，数据搬运随之减少。下面先比较 HBM 与 SRAM，再从稳定 Softmax 推导到完整的分块计算。

In [ ]:
# 本附录只用 torch 和 numpy，教学实现手写；后面用 SDPA 作参考
import math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print(f"torch: {torch.__version__}")


## 1. GPU 存储层次

GPU 存储不止两层，还有寄存器、L1、L2、共享内存等。理解 FlashAttention 时先比较两个层级：容量较大的 HBM，与计算附近容量较小的片上存储。

| 存储 | 示例容量 | 带宽与作用 |
|:---|:---|:---|
| A100 80GB HBM | 80 GB | 约 2 TB/s，存放模型与输入输出张量 |
| A100 每 SM 的 L1/共享内存 | 合计 192 KiB，其中共享内存最多 164 KiB/SM | 片上工作区，容量按 SM 限制，不能当作全 GPU 公共池 |
| 片上 SRAM 的教学带宽假设 | 不对应一个统一缓存池 | ~19 TB/s 是数量级示意，非所有算子的固定实测速率 |

以 19 / 2 = 9.5 比较数量级，有助于理解减少 HBM 访问的价值，但不能据此直接预测 9.5 倍加速。实际时间同时取决于计算、带宽、kernel 调度与形状。

当数据搬运限制吞吐时，称为 **memory-bound**。朴素 Attention 的中间矩阵读写可能成为瓶颈，FlashAttention 通过融合与分块减少这部分访问；不能把所有 Attention 配置都认定为相同瓶颈。

[A100 架构说明](https://developer.nvidia.com/blog/nvidia-ampere-architecture-in-depth/)。


### 标准 Attention 的数据搬运

标准 Attention 的 forward 分四步：

1. 计算 $S = QK^\top$，得到 $N \times N$ 的注意力分数矩阵，**写回 HBM**
2. 从 HBM 读回 $S$，算 row max，写回 HBM
3. 从 HBM 读回 $S$ 和 max，算 softmax 得到 $P$，**写回 HBM**
4. 从 HBM 读回 $P$ 和 $V$，算 $O = PV$，写回 HBM

先停一下，注意一个反直觉的地方：第 2 步的 max 和第 3 步的 softmax，其实都不需要把整个 $S$ 看一遍才算得出来。换句话说，完全可以边走边算，下一节的 online softmax 就是这么干的。这里分离 kernel 的示意会多次访问 $S$，并物化 $S$、$P$；实际实现可能融合 max 与 softmax，访问次数并不固定。

每个 $N \times N$ 矩阵都要在 HBM 上来回读写多次，这就是浪费的根源。下面用具体数字感受一下规模。

In [ ]:
# 标准注意力矩阵在不同序列长度下的显存占用（FP16，2 bytes/元素）
def attn_matrix_bytes(seq, head_dim=128, n_heads=32, bytes_per_elem=2):
    # 一次 forward 需要写回 HBM 的中间矩阵：S（attention scores）和 P（softmax 后）
    # 每个 head、每个 batch 都有一份
    elems = n_heads * seq * seq
    return elems * bytes_per_elem

def fmt(b):
    if b >= 1024**3:
        return f"{b/1024**3:.2f} GiB"
    if b >= 1024**2:
        return f"{b/1024**2:.1f} MiB"
    return f"{b/1024:.1f} KiB"

print(f"{'序列长度':>8}  {'S/P 矩阵单份':>15}  {'5 次读写总量':>20}")
print("-" * 50)
for seq in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
    single = attn_matrix_bytes(seq)
    # S 写 1 次、读 2 次（算 max、算 softmax），P 写 1 次、读 1 次
    traffic = single * 5
    print(f"{seq:>8d}  {fmt(single):>15}  {fmt(traffic):>15}")

print()
print("seq=32768、32 heads、单份 FP16 矩阵约 68.7 GB；5 次访问约 343.6 GB。")
print("按 2 TB/s 理想带宽估计下界约 0.17 秒，未计其他计算与访存。")


## 2. Softmax 的数值稳定性

进入 online softmax 之前，先解决一个更基础的问题：为什么 softmax 需要数值稳定？因为 softmax 公式 $\frac{e^{x_i}}{\sum_j e^{x_j}}$ 直接算会爆。

为什么会爆？因为指数函数 $e^x$ 是增长最陡的函数之一。具体来说，$x$ 每加 1，$e^x$ 就翻大约 2.72 倍。FP32 的 $e^{88}$ 仍是有限值，约超过 88.7 才溢出到 inf。

对 softmax 来说这尤其严重：公式里分子分母都要算 $e^{x_i}$。也就是说，**只要有一个 $x_i$ 达到 100，整个 softmax 就废了**——那个位置的分子变成 inf，分母也变成 inf，结果直接错乱。

而 attention 里的 $QK^\top$ 元素是两个 $d$ 维向量的内积，元素尺度取决于 Q、K 范数以及 $1/\sqrt d$ 缩放，不能仅由 $d=128$ 判断超过 100 是否常见。因此，用代码验证一下到底多容易爆。

> **关键观察**：softmax 的溢出不是「个别元素坏了」，而是「整个结果都废了」——因为分子分母同时变 inf，连 nan 都可能出现。所以数值稳定性不是锦上添花，而是 attention 能不能跑起来的必要条件。

In [ ]:
# 演示 exp 在 FP32 下的溢出阈值
import torch

x = torch.tensor([80.0, 85.0, 88.0, 90.0, 100.0])
print("x       exp(x)")
for xi in x:
    print(f"{xi.item():>6.1f}  {torch.exp(xi).item():>15.3e}")

print()
print("FP32 下 exp(89) 左右就接近 inf。")

# 模拟 attention score 的量级
d_head = 128
q = torch.randn(d_head)
k = torch.randn(d_head)
score = (q * k).sum() / math.sqrt(d_head)
print(f"随机初始化下 d=128 的 attention score 大约在 ±{abs(score.item()):.1f} 量级")
print("分数尺度取决于 Q/K 范数和缩放；此处不是训练后分布的测量。")


### 2.1 Stable Softmax

怎么让 $e^x$ 不爆？思路很朴素：既然指数一大就爆，那就先把指数变小一点再算。为什么平移有用？因为 softmax 有一个关键性质：**对输入整体平移一个常数 $c$，输出不变**。

$$
\mathrm{softmax}(x - c)_i = \frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{x_i} \cdot e^{-c}}{e^{-c} \sum_j e^{x_j}} = \mathrm{softmax}(x)_i
$$

$e^{-c}$ 在分子分母上同时出现，被约掉了。所以平移不改变结果。那么 $c$ 取多少最划算？把 $c$ 取成 $x_\max$，最大的指数变成 $e^0 = 1$，所有指数都 $\le 1$，不再有溢出风险。

这就是 stable softmax：先扫一遍找 max，再扫一遍算 $\frac{e^{x_i - x_\max}}{\sum_j e^{x_j - x_\max}}$。它解决了有限输入的指数溢出；扫描与访存是否融合还取决于实现。下一节把 max 与分母的统计合成一遍。

In [4]:
def naive_softmax(x):
    """直接实现，会溢出"""
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum()

def stable_softmax(x):
    """减去 max，数值稳定"""
    m = x.max()
    exp_x = torch.exp(x - m)
    return exp_x / exp_x.sum()

# 构造一个会让 naive 版爆炸的输入
x = torch.tensor([1.0, 2.0, 3.0]) * 50  # max 是 150
print("输入:", x.tolist())
print("naive softmax:", naive_softmax(x).tolist())
print("stable softmax:", stable_softmax(x).tolist())

print()
print("naive 版本里 exp(150) 溢出成 inf，最终结果变 nan。")
print("stable 版本里 max=150 被减掉，最大指数是 exp(0)=1，安全。")


输入: [50.0, 100.0, 150.0]
naive softmax: [0.0, nan, nan]
stable softmax: [3.783505853677006e-44, 1.9287498933537385e-22, 1.0]

naive 版本里 exp(150) 溢出成 inf，最终结果变 nan。
stable 版本里 max=150 被减掉，最大指数是 exp(0)=1，安全。


### 2.2 LogSumExp Trick

很多时候我们要算的是 $\log \sum_i e^{x_i}$（比如交叉熵损失里），它比 softmax 还容易爆。为什么单独算对数和会溢出？因为直接算 $\sum_i e^{x_i}$ 同样会溢出。

把 stable softmax 的思路反过来用：在 $\log$ 里把 max 提出来。也就是说，从指数里减去最大值再补偿。

$$
\log \sum_i e^{x_i} = \log \left( e^{x_\max} \sum_i e^{x_i - x_\max} \right) = x_\max + \log \sum_i e^{x_i - x_\max}
$$

减掉 max 之后，最大的指数项是 $e^0 = 1$，求和至少是 1，不会对 0 取对数（有限输入时）。这就是 logsumexp trick，几乎所有深度学习框架的 `log_softmax` 都基于它。

先记住这个套路：「算不了就先把 max 提出来」。第 3 节你会再见到它——online softmax 的 running 变量，本质就是在流式地维护这个 logsumexp。

In [ ]:
def naive_logsumexp(x):
    """直接 log(sum(exp(x)))，会溢出"""
    return torch.log(torch.exp(x).sum())

def stable_logsumexp(x):
    """logsumexp trick"""
    m = x.max()
    return m + torch.log(torch.exp(x - m).sum())

x = torch.tensor([100.0, 101.0, 102.0])
print("输入:", x.tolist())
print("naive logsumexp:", naive_logsumexp(x).item())
print("stable logsumexp:", stable_logsumexp(x).item())
print("torch 官方:", torch.logsumexp(x, dim=0).item())

print()
print("稳定实现与 torch 参考接近；naive 返回 inf。")
print("这个 trick 后面会在 online softmax 里反复出现——")
print("FlashAttention 的 running 变量本质上就是在维护 logsumexp 的流式版本。")


## 3. Online Softmax

stable softmax 解决了溢出问题，但代价是要扫两遍数据：一遍找 $x_\max$，一遍算 $\sum_j e^{x_j - x_\max}$。分离扫描可能增加读访存，但一行驻留片上时并不必然读两次 HBM——前面刚说少搬数据是关键，这两遍扫描恰恰是最该省的搬运。

那么，能不能只扫一遍就算出结果呢？max 和分母两个量都依赖全部数据，看起来必须先把数据全看完才能算。但其实不需要，我们可以在扫描过程中，每看到一个数就顺便更新「当前已知的 max」和「以当前 max 为基准的部分和」。

换句话说，一旦后面出现了更大的数，就把之前累加的结果按新标准换算一下。这样一遍扫描就能同时得到 max 和分母，计算这两个统计量不再回头读数据；若要输出全部概率，还需访问输入，Attention 可通过同时累积加权输出避免物化概率。

FlashAttention 的第一个核心技巧就来自这个观察：**边找 max 边累加分母，遇到更大的 max 时把旧累加结果 rescale 一遍**。这就是 online softmax。

### 3.1 两遍算法与单遍算法

下面描述经典 max 与分母统计的两遍流程；最终输出概率还需访问输入，具体如下方代码块所示。第一遍找最大值 m，第二遍累加分母 d，最后返回每个位置的 softmax 值。

```
第一遍：扫 x_1, ..., x_N，记录 m = max
第二遍：扫 x_1, ..., x_N，累加 d = sum exp(x_i - m)
返回：softmax_i = exp(x_i - m) / d
```

那么，online softmax 怎么改成单遍呢？它一边扫描一边记一本「账」，账本上只维护两个数，如第二个代码块所示。

```
扫描 x_1, ..., x_N，同时维护：
  m_k = max(x_1, ..., x_k)            # 目前见过的最大值（running max）
  d_k = sum_{j=1}^k exp(x_j - m_k)    # 以当前 m_k 为基准的累加分母（running 分母）

每来一个新的 x_{k+1}：
  1. 更新 max: m_{k+1} = max(m_k, x_{k+1})
  2. rescale 旧累加: d_{k+1} = d_k * exp(m_k - m_{k+1}) + exp(x_{k+1} - m_{k+1})
```

关键在第二步的更新规则。为什么多出 `exp(m_k - m_{k+1})` 这个因子？因为 max 从 $m_k$ 涨到 $m_{k+1}$，记账标准变了，之前以旧标准算的每一项 $\exp(x_j - m_k)$ 都得换算成新标准 $\exp(x_j - m_{k+1})$。

换算公式是 $\exp(x_j - m_{k+1}) = \exp(x_j - m_k) \cdot \exp(m_k - m_{k+1})$。因此不用逐个重算，只要给 $d_k$ 整体乘一个 $\exp(m_k - m_{k+1})$ 就完成了全部换算。

当 max 没变（$m_k = m_{k+1}$）时，$\exp(m_k - m_{k+1}) = \exp(0) = 1$。换句话说，账本不用动，直接把新项加进去即可。

### 3.2 Online Softmax 手算

公式看多了容易晕，我们拿具体例子把账本走一遍。用 $x = [3, 1, 2]$ 演示，最终的 stable softmax 是 $\frac{[e^3, e^1, e^2]}{e^3 + e^1 + e^2}$，max=3。

按 online 流程一步步走，下面这张表就是我们刚才说的那本流水账：

| 步骤 | $x_{k+1}$ | 新 max $m$ | rescale 因子 $e^{m_\text{old} - m_\text{new}}$ | 新分母 $d$ |
|:---|:---|:---|:---|:---|
| 初始 | - | $-\infty$ | - | 0 |
| 看到 $x_1=3$ | 3 | $\max(-\infty, 3) = 3$ | $e^{-\infty - 3} = 0$ | $0 \cdot 0 + e^{3-3} = 1$ |
| 看到 $x_2=1$ | 1 | $\max(3, 1) = 3$ | $e^{3-3} = 1$ | $1 \cdot 1 + e^{1-3} = 1 + 0.135 = 1.135$ |
| 看到 $x_3=2$ | 2 | $\max(3, 2) = 3$ | $e^{3-3} = 1$ | $1.135 \cdot 1 + e^{2-3} = 1.135 + 0.368 = 1.503$ |

逐行读一遍这张表。第一行（$x_1=3$）max 从 $-\infty$ 涨到 3，旧账清零（rescale 因子为 0），新账记下 $e^0 = 1$。后面 max 一直是 3 不变，rescale 因子恒为 1，因此每来一个新数只需把 $e^{x_i - 3}$ 加进账里。

最终得到 $m=3$、$d=1.503$。验算一下：$e^3 + e^1 + e^2 = 20.09 + 2.72 + 7.39 = 30.20$，而 $d \cdot e^m = 1.503 \cdot e^3 = 1.503 \cdot 20.09 = 30.20$。两者相等，说明 online 算出来的 $(m, d)$ 和经典算法完全一致。

注意整个过程只扫了一遍 $x$，每一步都只用当前元素和两个 running 变量，从不回头重读旧数据。这就是单遍算法的意义。

In [ ]:
def online_softmax(x):
    """一遍计算 normalizer，再输出所有概率"""
    m = float('-inf')   # running max
    d = 0.0             # running 分母 sum exp(x_i - m)

    for xi in x:
        m_new = max(m, xi.item())
        # rescale 旧累加，加上新项
        d = d * math.exp(m - m_new) + math.exp(xi.item() - m_new)
        m = m_new

    # 第二遍：用最终的 (m, d) 算每个位置的 softmax
    # 这一步 FlashAttention 会通过 tiling 融合进 forward
    return torch.tensor([math.exp(xi.item() - m) / d for xi in x])

x = torch.tensor([3.0, 1.0, 2.0])
print("输入:", x.tolist())
print("online softmax:    ", online_softmax(x).tolist())
print("stable softmax:    ", stable_softmax(x).tolist())
print("torch 官方 softmax:", torch.softmax(x, dim=0).tolist())

print()
print("三者数值完全相同——online 流程没有引入任何近似。")


## 4. 从 softmax 到 Attention 输出

等一下——我们费劲算 softmax 是为了什么？为了给每个 value 一个权重。attention 的最终输出不是 softmax 本身，而是 $o = \sum_i p_i v_i$，其中 $p_i$ 是 softmax 权重，$v_i$ 是 value 向量。

把 $p_i$ 展开，得到如下等式：

$$
o = \sum_i \frac{e^{x_i}}{\sum_j e^{x_j}} v_i = \frac{\sum_i e^{x_i} v_i}{\sum_i e^{x_i}}
$$

**关键观察来了**：这个式子里的 $x_\max$ 其实会约掉。分子分母同乘 $e^{-x_\max}$，就变成下方公式：

$$
o = \frac{\sum_i e^{x_i - x_\max} v_i}{\sum_i e^{x_i - x_\max}}
$$

两个求和里减的是同一个 $x_\max$，正好是第 2 节 stable softmax 的套路——数值稳定，输出不变。这里 $x_i = q \cdot k_i / \sqrt{d}$ 是 query 和第 $i$ 个 key 的点积，$v_i$ 是第 $i$ 个 value。

所以只要在账本上再加一列：除了 $(m, d)$，再维护一个 running 输出 $o_k = \sum_{i=1}^k e^{x_i - m_k} v_i$。更新规则和 $d$ 完全对称，因为本质都是「$e^{x_i - m}$ 的加权和」，如公式：

$$
o_{k+1} = o_k \cdot e^{m_k - m_{k+1}} + e^{x_{k+1} - m_{k+1}} v_{k+1}
$$

扫完所有 $N$ 个 key 之后，真正的 attention 输出就是 $o_N / d_N$。

### 4.1 Attention 输出手算

用一个具体的 query 和三个 key-value 对，验证 online 流程的正确性。设 $d=1$ 简化数字（实际 $d=64$ 或 $128$），参数如下：

- $q = 1$，三个 key $k = [3, 1, 2]$，对应 value $v = [10, 20, 30]$
- attention score $x_i = q \cdot k_i = [3, 1, 2]$（省略 $\sqrt{d}$ 因为 $d=1$）
- 真实输出 $o = \frac{e^3 \cdot 10 + e^1 \cdot 20 + e^2 \cdot 30}{e^3 + e^1 + e^2}$

按 online 流程走。注意观察：$d$ 列和 $o$ 列的更新规则完全一样，只是 $o$ 每一项多乘一个对应的 $v_i$。表格如下：

| 步骤 | $x_{k+1}$ | $v_{k+1}$ | $m$ | rescale | $d$ | $o$ |
|:---|:---|:---|:---|:---|:---|:---|
| 初始 | - | - | $-\infty$ | - | 0 | 0 |
| $k=1$ | 3 | 10 | 3 | 0 | $e^0 = 1$ | $e^0 \cdot 10 = 10$ |
| $k=2$ | 1 | 20 | 3 | 1 | $1 + e^{-2} = 1.135$ | $10 + e^{-2} \cdot 20 = 10 + 2.71 = 12.71$ |
| $k=3$ | 2 | 30 | 3 | 1 | $1.135 + e^{-1} = 1.503$ | $12.71 + e^{-1} \cdot 30 = 12.71 + 11.04 = 23.75$ |

最终得到 $o / d = 23.75 / 1.503 = 15.80$。直接按定义算 $\frac{20.09 \cdot 10 + 2.72 \cdot 20 + 7.39 \cdot 30}{30.20} = \frac{200.9 + 54.4 + 221.7}{30.20} = \frac{477.0}{30.20} = 15.80$。两者相等，证明 online 流程正确。

In [ ]:
def online_attention(q, K, V):
    """
    单遍 attention，每个 query 独立处理

    参数:
        q: 一个 query 向量，shape [d]
        K: 所有 key，shape [N, d]
        V: 所有 value，shape [N, d]
    返回:
        attention 输出，shape [d]
    """
    d_head = q.shape[0]
    N = K.shape[0]

    m = float('-inf')       # running max
    d = 0.0                 # running 分母
    o = torch.zeros_like(q) # running 输出

    for i in range(N):
        # 当前 key 与 query 的点积（除以 sqrt(d) 控制分数尺度）
        x_i = torch.dot(q, K[i]).item() / math.sqrt(d_head)
        v_i = V[i]

        m_new = max(m, x_i)
        rescale = math.exp(m - m_new)

        d = d * rescale + math.exp(x_i - m_new)
        o = o * rescale + math.exp(x_i - m_new) * v_i
        m = m_new

    return o / d

# 手算例子：d=1，q=1，k=[3,1,2]，v=[10,20,30]
q = torch.tensor([1.0])
K = torch.tensor([[3.0], [1.0], [2.0]])
V = torch.tensor([[10.0], [20.0], [30.0]])
print("online attention 输出:", online_attention(q, K, V).item())
print("手算结果: 15.80")

# 对照标准实现
def standard_attention(q, K, V):
    d_head = q.shape[0]
    scores = (K @ q) / math.sqrt(d_head)
    weights = torch.softmax(scores, dim=0)
    return weights @ V

print("standard attention 输出:", standard_attention(q, K, V).item())


## 5. Tiling

到这一步，我们已经能在 CPU 上用单遍循环算 attention。但是 GPU 不是这样工作的——GPU 靠几万个核心并行工作，你让它一个 token 一个 token 串行循环，等于几万个核心都在等你，这是最差的形式。

那么 FlashAttention 的第二个核心技巧是什么？是 tiling：把 Q、K、V 切成块，让每块刚好能放进 SRAM，循环在块级别进行。不同 Q 块可独立并行，同一 Q 块遍历 K/V 时仍维护 running 状态；块内部用矩阵运算一把算完。

### 5.1 分块结构

设序列长度 $N$、head 维度 $d$、block size $B$（通常取 64 或 128）。把序列切成 $N / B$ 块，具体切分方式如下：

- 外层循环遍历 Q 的块：$Q_1, Q_2, \dots, Q_{N/B}$，每块 shape $[B, d]$
- 内层循环遍历 K、V 的块：$K_1, K_2, \dots, K_{N/B}$，每块 shape $[B, d]$

对每一个 Q 块，独立维护一组 running 变量 $(m, d, o)$，shape 分别是 $[B]$、$[B]$、$[B, d]$。在内层循环里，每读入一对 $(K_j, V_j)$ 块就更新一次，伪代码如下：

```
for each Q block i (outer loop, parallelizable across blocks):
    m_i = -inf, d_i = 0, o_i = 0   # shape [B], [B], [B, d]
    for each K, V block j (inner loop):
        load K_j, V_j from HBM to SRAM
        S = Q_i @ K_j^T            # [B, B]，只在 SRAM 里存在
        m_block = S.max(dim=-1)    # [B]
        m_new = max(m_i, m_block)
        P = exp(S - m_new[:, None])
        d_i = d_i * exp(m_i - m_new) + P.sum(dim=-1)
        o_i = o_i * exp(m_i - m_new)[:, None] + P @ V_j
        m_i = m_new
    write o_i / d_i back to HBM   # 只写最终结果
```

注意伪代码里的 `S` 和 `P`：它们都是 $B \times B$ 的小块，**从头到尾只存在于 SRAM，从未被写回 HBM**。每个 $S_{ij}$ 算出来就用掉了，下一块来了直接覆盖。换句话说，这就是全部节省数据的秘密——标准实现把整张 $N \times N$ 中间矩阵写回 HBM；这里的小 $B \times B$ 中间块留在片上，Q/K/V 的 HBM 访问仍然存在。

### 5.2 Tiling 的计算等价性

online softmax 的更新规则是**可结合的**：先在块内累加，再用块级 max rescale，和一次性扫完所有元素结果相同。原因是更新规则只依赖三个量 $(m, d, o)$，块的边界对最终结果没有影响。

直觉上：把 $N$ 个 key 分成两半先各自算 $(m_1, d_1, o_1)$ 和 $(m_2, d_2, o_2)$，再用全局 max $m = \max(m_1, m_2)$ rescale 合并，等价于一次性扫完。因此，这就是块级 online softmax 的数学保证。

> **关键观察**：tiling 之所以不破坏正确性，靠的正是 online softmax 的更新规则只依赖 $(m, d, o)$ 三个量——块边界在哪、怎么切，在精确算术下对最终结果没有影响；浮点舍入与 kernel 约束仍需考虑。这也是 FlashAttention 能安全地在硬件与实现支持的 block size 下运行的底气。

### 5.3 Block Size 的选择

分块要符合每 SM 的共享内存、寄存器和 occupancy 限制，不能拿全 GPU 的片上容量给单个 block 使用。

粗略同时保留 $Q_i,K_j,V_j$ 与 $S,P$，需要 $3Bd+2B^2$ 个元素。B=d=128 时，$3\cdot128\cdot128=49152$，$2\cdot128^2=32768$，合计 81920 个 FP16 元素，即 160 KiB。

A100 每 SM 共享内存最多 164 KiB，这个粗略存储已经接近容量，不能据此声称同一 SM 能同时装下多个这样的 block。实际 kernel 会分配到共享内存或寄存器、复用缓冲区，并使用不同累积精度。

B 太小会增加循环与访存调度开销，不等于每次内循环都启动一个新 kernel；B 太大可能降低 occupancy 或超出资源限制。生产实现通常按硬件、head 维度与数据类型选择配置。


## 6. Exact Attention

为什么要单独讨论「精确性」？因为有一个常见误解：很多人以为 FlashAttention 为了快而牺牲了精度，这一节就是要澄清它并没有。

我们先看两种真正牺牲精度的做法，作为对照。第一种是 sparse attention（比如 Longformer、Big Bird），它主动跳过一部分 query-key 对，认为「远距离的注意力不重要」，这属于**结构上的近似**——输出和 dense attention 数学上不同。第二种是 linear attention（Performer、Linear Transformer），它用 kernel trick 把 $QK^\top$ 替换成可分解的核函数，把复杂度从 $O(N^2)$ 降到 $O(N)$，这也是**数学上的近似**。

FlashAttention 不一样，它只是**重新安排了计算顺序**：

- dense 版本：先算完整 $S$，再 softmax，再乘 $V$
- flash 版本：分块算 $S$ 的一小块，立刻 softmax，立刻乘 $V$ 的一小块，running 累加

换句话说，两者的数学运算完全等价，只是中间结果出现的位置不同。FlashAttention 永远不把完整 $S$ 写入 HBM，但每一个 $S_{ij}$ 都参与了计算，数学定义相同，浮点差异取决于 dtype、输入与归约顺序；本例 FP32 小规模测试用 $10^{-5}$ 容差。替换时仍需核对 mask、dropout 与支持的形状，也不能承诺训练轨迹逐位相同。

> **关键观察**：区分「近似」和「重排」的判据很简单——把中间结果完整保留下来，两种算法是否给出相同结果。sparse / linear attention 会给出不同结果，FlashAttention 不会，区别就在这里。

## 7. 简化 PyTorch 实现

为什么要用纯 PyTorch 写一个教学版 FlashAttention？目的是**不追求性能**（Python 循环比 CUDA 慢得多），只追求数学正确，让读者看清 online softmax + tiling 的每一步在做什么。

读这段代码时，请对照第 5.1 节的伪代码。你会发现外层 Q 块循环、内层 K/V 块循环、每块更新一次 $(m, d, o)$，都是一一对应的。

实现要点：

- 单 head、batch=1 简化（多 head 只是外层循环）
- 外层 Q 块循环，内层 K/V 块循环
- 用 `torch.einsum` 或 `@` 处理块级矩阵乘法
- 显式维护 $(m, d, o)$ 三个 running 变量

In [ ]:
def flash_attention_forward(Q, K, V, block_size=4, causal=False):
    """
    教学版 FlashAttention forward，单 head，batch=1

    参数:
        Q: shape [N, d]
        K: shape [N, d]
        V: shape [N, d]
        block_size: Q/K/V 分块大小
        causal: 是否因果 mask（下三角）
    返回:
        O: shape [N, d]
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    # 这是 CPU 默认张量的数学演示，不模拟 SRAM 驻留
    # 生产 kernel 使用寄存器与共享内存等片上资源
    O = torch.zeros(N, d)

    # 外层循环：每个 Q 块独立处理
    n_blocks = (N + block_size - 1) // block_size
    for i in range(n_blocks):
        q_start = i * block_size
        q_end = min(q_start + block_size, N)
        Qi = Q[q_start:q_end] * scale  # [B_q, d]，提前乘 scale

        # 这个 Q 块的 running 变量
        B_q = q_end - q_start
        m_i = torch.full((B_q,), float('-inf'))
        d_i = torch.zeros(B_q)
        o_i = torch.zeros(B_q, d)

        # 内层循环：遍历所有 K/V 块
        for j in range(n_blocks):
            k_start = j * block_size
            k_end = min(k_start + block_size, N)
            Kj = K[k_start:k_end]  # [B_k, d]
            Vj = V[k_start:k_end]  # [B_k, d]

            # 算这一块的 attention score（教学版为普通 PyTorch 中间张量）
            S = Qi @ Kj.transpose(-2, -1)  # [B_q, B_k]

            # 因果 mask：query 位置 < key 位置的地方置 -inf
            if causal:
                q_idx = torch.arange(q_start, q_end)[:, None]
                k_idx = torch.arange(k_start, k_end)[None, :]
                mask = k_idx > q_idx
                S = S.masked_fill(mask, float('-inf'))

            # online softmax 更新
            m_block = S.max(dim=-1).values      # [B_q]
            m_new = torch.maximum(m_i, m_block)

            # 旧累加 rescale
            alpha = torch.exp(m_i - m_new)      # [B_q]
            # 新块的 exp（减去新 max 保证数值稳定）
            P = torch.exp(S - m_new[:, None])   # [B_q, B_k]

            d_i = d_i * alpha + P.sum(dim=-1)
            o_i = o_i * alpha[:, None] + P @ Vj
            m_i = m_new

        # 这个 Q 块的最终输出
        O[q_start:q_end] = o_i / d_i[:, None]

    return O

# 小例子：seq=8, d=4，手算友好
torch.manual_seed(42)
N, d = 8, 4
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

O_flash = flash_attention_forward(Q, K, V, block_size=4)
print("FlashAttention 输出 (前 3 行):")
print(O_flash[:3])


## 8. 数值等价性验证

为什么还要做验证？因为光实现不够，我们必须证明这个教学版和标准形式 attention 数值相同。

对照两种参考实现：

1. 从零实现的标准 attention（显式构造 $N \times N$ 矩阵）
2. PyTorch 官方的 `scaled_dot_product_attention`（按设备、dtype、形状与后端开关选择可用实现，不保证选到 FlashAttention）

数值对拍检验本例实现，不能替代前面的等价推导。这里以 $10^{-5}$ 为测试容差，具体误差以输出为准。这正是第 6 节「exact 不是近似」的实验证据。

In [9]:
def standard_attention(Q, K, V, causal=False):
    """标准 attention，显式构造 N×N 矩阵"""
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    S = (Q @ K.T) * scale  # [N, N]
    if causal:
        mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
        S = S.masked_fill(mask, float('-inf'))
    P = torch.softmax(S, dim=-1)
    return P @ V

# 非因果场景
O_std = standard_attention(Q, K, V, causal=False)
O_sdpa = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=False
)[0]

print("=== 非因果场景 ===")
print(f"Flash vs 标准实现 最大差: {(O_flash - O_std).abs().max().item():.2e}")
print(f"Flash vs SDPA     最大差: {(O_flash - O_sdpa).abs().max().item():.2e}")
print(f"标准  vs SDPA     最大差: {(O_std - O_sdpa).abs().max().item():.2e}")

# 因果场景
O_flash_causal = flash_attention_forward(Q, K, V, block_size=4, causal=True)
O_std_causal = standard_attention(Q, K, V, causal=True)
O_sdpa_causal = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=True
)[0]

print()
print("=== 因果场景 ===")
print(f"Flash vs 标准实现 最大差: {(O_flash_causal - O_std_causal).abs().max().item():.2e}")
print(f"Flash vs SDPA     最大差: {(O_flash_causal - O_sdpa_causal).abs().max().item():.2e}")

print()
print("所有差异都在 1e-6 量级——浮点误差范围内，FlashAttention 数学等价。")


=== 非因果场景 ===
Flash vs 标准实现 最大差: 8.94e-08
Flash vs SDPA     最大差: 1.79e-07
标准  vs SDPA     最大差: 1.19e-07

=== 因果场景 ===
Flash vs 标准实现 最大差: 1.19e-07
Flash vs SDPA     最大差: 1.19e-07

所有差异都在 1e-6 量级——浮点误差范围内，FlashAttention 数学等价。


In [10]:
# 更大的测试：seq=256，d=64，看长序列下也保持精度
torch.manual_seed(0)
N, d = 256, 64
Q_big = torch.randn(N, d)
K_big = torch.randn(N, d)
V_big = torch.randn(N, d)

O_flash_big = flash_attention_forward(Q_big, K_big, V_big, block_size=64)
O_sdpa_big = F.scaled_dot_product_attention(
    Q_big[None], K_big[None], V_big[None]
)[0]

max_diff = (O_flash_big - O_sdpa_big).abs().max().item()
mean_diff = (O_flash_big - O_sdpa_big).abs().mean().item()
print(f"seq=256, d=64:")
print(f"  最大差:  {max_diff:.2e}")
print(f"  平均差:  {mean_diff:.2e}")
print(f"  通过 1e-5 阈值: {max_diff < 1e-5}")

# 顺便看 block size 对结果的影响（应该完全不影响）
print()
print("block size 对数值正确性的影响（应全部 < 1e-5）：")
for bs in [16, 32, 64, 128]:
    O_bs = flash_attention_forward(Q_big, K_big, V_big, block_size=bs)
    diff = (O_bs - O_sdpa_big).abs().max().item()
    print(f"  block_size={bs:>3d}: 最大差 {diff:.2e}")


seq=256, d=64:
  最大差:  7.75e-07
  平均差:  2.82e-08
  通过 1e-5 阈值: True

block size 对数值正确性的影响（应全部 < 1e-5）：


  block_size= 16: 最大差 4.77e-07


  block_size= 32: 最大差 5.96e-07


  block_size= 64: 最大差 7.75e-07
  block_size=128: 最大差 1.01e-06


## 9. MoE、专家分组与变长 Attention

常见 Transformer MoE 将 **FFN 换成专家组**，Attention 仍按原序列计算；并非每个 FFN expert 内都有一层 self-attention。专家路由后的 token 分组，通常用 grouped GEMM 处理 FFN，而不是直接作为新的 Attention 序列。

### 9.1 专家收到多少 token？

设 batch 为 B、序列长 S、专家数 E、top-k 为 k，均衡情况下每专家平均收到 $BSk/E$ 个 token：

- B=1、S=4096、k=2、E=8：平均 1024 个；
- B=1、S=4096、k=1、E=64：平均 64 个。

这些 token 来自不同位置或样本。如果把它们直接拼成序列做 Attention，就改变了原来的上下文关系。

### 9.2 专家缓冲区的 padding 账本

为教学比较，假设每专家都预留 BS 个 token 槽，有效比例为 $k/E$。E=8、k=2 时是 25%。这只是一种故意过度 padding 的存储/逐 token 工作账本，实际实现会按容量、动态路由与负载均衡安排缓冲区。

| 专家 FFN 输入布局 | token 槽开销 | 说明 |
|:---|:---|:---|
| 每专家预留 BS 槽 | 有效比例 k/E | 本例的粗略 padding 假设 |
| 按有效 token 紧凑分组 | 避免上述无效槽 | 用 grouped GEMM 处理专家 FFN；仍有调度、对齐与不均衡开销 |

### 9.3 FlashAttention varlen 解决什么？

varlen 用累计长度 `cu_seqlens` 描述不同真实序列的边界，适合减少变长序列 padding。它不自动解决 MoE FFN 专家路由；把专家 ID 当作序列边界通常会改变 Attention 的数学定义。

FlashAttention 的输入仍是正确的 Q/K/V 和 Attention 边界。dense 模型与常见 MoE 都可以使用标准或 varlen Attention；是否使用 varlen 取决于序列布局，而非是否有专家。下面保留原配置，计算上述**token 槽效率**，不将 E/k 当作实测 Attention 加速比。


In [ ]:
# 专家缓冲区 token 槽账本，不是 Attention FLOPs
# n_heads、d 为兼容保留，不影响槽数。
def moe_padding_waste(B, S, E, k, n_heads, d):
    avg_tokens_per_expert = B * S * k / E
    actual_tokens = B * S
    eff_ratio = avg_tokens_per_expert / actual_tokens
    reserved_slots = E * actual_tokens
    compact_slots = E * avg_tokens_per_expert
    return eff_ratio, reserved_slots, compact_slots

print(f"{'配置':>30}  {'有效槽比例':>18}  {'槽数比':>12}")
print("-" * 65)
configs = [
    ("dense (E=1, k=1)", 1, 4096, 1, 1, 32, 128),
    ("MoE E=8, k=2", 1, 4096, 8, 2, 32, 128),
    ("MoE E=64, k=1", 1, 4096, 64, 1, 32, 128),
    ("MoE E=64, k=2", 1, 4096, 64, 2, 32, 128),
    ("MoE E=128, k=2", 1, 4096, 128, 2, 32, 128),
]
for name, B, S, E, k, nh, d in configs:
    eff, _, _ = moe_padding_waste(B, S, E, k, nh, d)
    waste = 1 / eff if eff > 0 else float('inf')
    print(f"{name:>30}  {eff:>14.1%}  {waste:>9.1f}x")

print()
print("E=128、k=2：有效槽比例 1.6%，预留/紧凑槽数比为 64。")
print("这里的理论槽数比不是实测 Attention 加速比。")


## 附录：FlashAttention 的演进

前面我们用纯 Python 复现了 FlashAttention 的核心思想。下面介绍 FlashAttention-1/2/3 的三次代表性改进，每一代都在解决一个具体的硬件问题。顺着这条线看下来，你会更清楚算法设计是如何适配硬件特性的。

### A7.1 FlashAttention-2 的计算优化

为什么要讲 FA-2？因为 FlashAttention-1 的 forward 已经很快，但反向传播和 work partitioning 还有改进空间。FlashAttention-2（Dao 2023）针对三点做了优化。

第一，**减少非 matmul FLOPs**。GPU 上 matmul（tensor core）的吞吐远高于其他运算（比如 rescale、softmax）。FA-1 在 inner loop 里有一些非 matmul 操作，FA-2 把它们重新安排到 matmul 的间隙，让 tensor core 的利用率更高。

第二，**更好的并行化**。FA-1 主要在 batch 和 head 维度并行，序列长但 batch 小的场景下 GPU 利用率低。FA-2 把序列维度也拆开并行——不同的 Q 块分到不同 SM 上，长序列场景下吞吐显著提升。

第三，**work partitioning**。FA-1 的 inner/outer loop 划分让 GPU 的 warp 之间频繁同步。FA-2 重新设计让每个 warp 独立处理一块，减少同步开销。实测结果说明改进有多大：FA-2 比 FA-1 快约 2 倍，论文配置达到 A100 理论峰值的 50–73%，不代表所有输入形状。

### A7.2 FlashAttention-3、Hopper 与 FP8

FlashAttention-3（Shah et al. 2024）专门为 Hopper 架构（H100）设计，利用三个新特性来解决硬件瓶颈。第一，**WGMMA（warp-group matrix multiply accumulate）**。Hopper 引入的异步 tensor core 指令，matmul 可以和其他运算 overlap。FA-3 把 softmax 和 rescale 安排在 matmul 的间隙里执行，几乎完全隐藏非 matmul 开销。

第二，**TMA（tensor memory accelerator）**。Hopper 的硬件 DMA，可以异步在 HBM 和 SRAM 之间搬数据，减少搬运所需的 SM 指令工作，而非完全不占任何 SM 资源。FA-3 用 TMA 做双层 buffering——下一块 K/V 在算当前块时就已经预取。

第三，**FP8 tensor core**。Hopper 支持 FP8（E4M3 / E5M2）的 tensor core matmul，吞吐是 FP16 的两倍。FA-3 在 forward 用 FP8 算 $QK^\top$ 和 $PV$，但 softmax 部分仍用 FP32 累加保证数值稳定。性能上，实测 FA-3 在 H100 上达到 论文报告 FP16 最高约 740 TFLOPs/s（75% 利用率），FP8 接近 1.2 PFLOPs/s；这两组数字不能混用。

### A7.3 Triton 实现导读

OpenAI 的 Triton 是 Python 写 GPU kernel 的高级语言，比 CUDA 更易读。仓库还保留一份早期实验性 Triton 实现，阅读时要注意其版本与限制。

源码位于 [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) 仓库的 `flash_attn/flash_attn_triton.py`。核心结构是一个 `@triton.jit` 装饰的函数，Q 块通过 program ID 分配，K/V 块在 kernel 内循环遍历，每个块用 `tl.load` 从 HBM 读、用 `tl.store` 写回。online softmax 的 $(m, d, o)$ 更新直接翻译成几行 Triton 代码，和本附录实现的 PyTorch 版逻辑完全对应。

阅读建议：先看本附录的 PyTorch 实现建立直觉，再对照 Triton 版理解 GPU 编程细节（shared memory、warp 同步、block 指针推进）。Triton 官方教程也有一份 [Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html) 的简化版本，比 Dao 官方版更易读。

### 参考文献

- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022
- Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691), 2023
- Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024
- Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018（online softmax 的原始论文）
- Tillet et al., [Triton: an intermediate language and compiler for tiled neural network computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), 2019

## 小结

FlashAttention 将存储层级、数值稳定与并行布局联系起来：避免把完整 Attention 中间矩阵写回 HBM，而不是省掉 dense Attention 的点积计算。

- [ ] GPU 有寄存器、L1、L2、共享内存与 HBM；片上资源按 SM 限制。
- [ ] 朴素 Attention 物化 $N\times N$ 的 S、P，多次读写可能限制吞吐，具体次数取决于融合。
- [ ] FP32 的 exp 在约 88.7 以上溢出；减去 max 是有限输入的稳定计算基础。
- [ ] LogSumExp：$\log\sum e^{x_i}=m+\log\sum e^{x_i-m}$。
- [ ] Online normalizer 一遍维护 running $(m,d)$，max 改变时用 $e^{m_{old}-m_{new}}$ rescale。
- [ ] 同时累积加权分子 $o$，最后以 $o/d$ 得到 Attention 输出。
- [ ] 不同 Q 块可以并行；同一块遍历 K/V 时维护状态，块级更新保持数学等价。
- [ ] Exact 指相同 Attention 定义，浮点误差仍与 dtype、输入和顺序有关。
- [ ] 常见 MoE 路由 FFN 专家；grouped GEMM 与 Attention varlen 的作用不同。
- [ ] FA-2 改进 work partitioning，FA-3 利用 Hopper WGMMA、TMA 与 FP8。


## 作业

下面三道作业帮你巩固 online softmax 和 attention 的核心更新规则。

> 可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」。


### 作业 1：实现 running max 更新

为什么布置这个作业？因为只看书不敢动手，就不知道 online softmax 的 running max 到底怎么更新。下面是一个简化版的 online softmax 框架，它缺少 running max 和 rescale 部分。

换句话说，你需要补全 `m_new`、`rescale`、`d` 三行，使输出和 `torch.softmax` 完全一致。



**小提示**：参考第 3.1 节的更新规则——新 max 是旧 max 和当前值的较大者，rescale 因子是 $\exp(m_\text{old} - m_\text{new})$，分母是「旧分母乘 rescale」加上「新项 $\exp(x_i - m_\text{new})$」。

In [ ]:
def online_softmax_homework(x):
    m = float('-inf')
    d = 0.0
    for xi in x:
        xi = xi.item()
        # TODO: 补全三行
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        m = m_new
    return torch.tensor([math.exp(xi - m) / d for xi in x])

x = torch.tensor([1.0, 5.0, 3.0, 7.0, 2.0])
result = online_softmax_homework(x)
expected = torch.softmax(x, dim=0)
assert torch.allclose(result, expected, atol=1e-6), f"差值过大: {(result - expected).abs().max()}"
print("作业 1 通过：online softmax 实现正确")

### 作业 2：验证 online attention 的数值等价

为什么验证数值等价？因为只有确认自己写的 attention 和库函数结果一致，才能放心使用。补全下面的函数，让它对随机输入 $(Q, K, V)$ 的输出和 `F.scaled_dot_product_attention` 差异小于 $10^{-5}$。

也就是说，你要保证误差极小。那么更新规则怎么写呢？



**小提示**：$o$ 的更新规则和 $d$ 完全对称——都是「旧值乘 rescale 因子」加上「$\exp(x_i - m_\text{new})$ 乘以对应的 $v_i$」。参考第 4 节的公式。

In [ ]:
def online_attention_homework(q, K, V):
    """单 query 版 attention"""
    d_head = q.shape[0]
    m = float('-inf')
    d = 0.0
    o = torch.zeros_like(q)

    for i in range(K.shape[0]):
        x_i = (q * K[i]).sum().item() / math.sqrt(d_head)
        v_i = V[i]
        # TODO: 补全 m_new、rescale、d、o 四步更新
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        # 4. o = ?
        m = m_new

    return o / d

torch.manual_seed(0)
d = 8
N = 16
q = torch.randn(d)
K = torch.randn(N, d)
V = torch.randn(N, d)
out = online_attention_homework(q, K, V)
ref = F.scaled_dot_product_attention(q[None, None], K[None], V[None])[0, 0]
assert torch.allclose(out, ref, atol=1e-5), f"差值: {(out - ref).abs().max()}"
print("作业 2 通过：online attention 数值等价")

### 作业 3：估算专家缓冲区的 token 槽效率

沿用第 9 节的教学假设：E=64 个专家、top-1（k=1）、S=8192、B=4，每专家都预留完整 BS 个槽。这里计算缓冲区/逐 token 工作量，**不是专家内部 Attention**。

总有效 token 分配为 BSk=4×8192×1=32768，每专家均衡平均为 32768/64=512。总预留槽为 EBS=64×32768=2097152，有效比例为 512/32768=1/64。紧凑布局把槽数减少 64 倍；这不等于实际 kernel 或 Attention 耗时加速 64 倍。

(1) 每专家平均多少有效 token？(2) 有效分配数 / 总预留槽数是多少？(3) 预留槽数 / 紧凑槽数是多少倍？

**提示**：平均 $BSk/E$；效率 k/E，槽数比 E/k。代码为兼容保留 `speedup_varlen` 变量名，但它在本题仅表示理论槽数比，不是测速结果。


In [ ]:
B, S, E, k = 4, 8192, 64, 1
total_tokens = B * S
# TODO: 补全三行
# tokens_per_expert = ?
# efficiency = ?       # 有效 / 总处理
# speedup_varlen = ?   # 预留槽 / 紧凑槽的理论比，非测速

assert tokens_per_expert == 512, f"tokens_per_expert 应该是 512，你得到 {tokens_per_expert}"
assert abs(efficiency - 1/64) < 1e-6, f"efficiency 应该是 1/64"
assert abs(speedup_varlen - 64) < 1e-6, f"speedup_varlen 应该是 64"
print("作业 3 通过：理解 MoE padding 浪费")
